# Experiment Bot — Analysis Notebook

This notebook loads human behavioral data and bot simulation output, computes per-subject metrics for Stop Signal and Stroop tasks, and provides a cross-platform comparison scaffold for when bot runs are available.

## 1. Setup

Imports, path constants, and helper utilities.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Project root — two levels up from scripts/
ROOT = Path('..').resolve()

HUMAN_DIR = ROOT / 'data' / 'human'
OUTPUT_DIR = ROOT / 'output'

STOP_SIGNAL_CSV = HUMAN_DIR / 'stop_signal.csv'
STROOP_CSV      = HUMAN_DIR / 'stroop.csv'

print(f'ROOT       : {ROOT}')
print(f'HUMAN_DIR  : {HUMAN_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')

In [ ]:
def load_bot_runs(task_prefix: str) -> list[dict]:
    """Scan output/ for completed bot runs matching task_prefix.

    A run is considered complete if its directory contains bot_log.json.
    Returns a list of parsed JSON dicts, one per run.

    Args:
        task_prefix: Subdirectory prefix to match, e.g. 'stopit_stop_signal'
                     or 'cognitionrun_stroop'. Pass '' to load all runs.
    """
    runs = []
    if not OUTPUT_DIR.exists():
        return runs
    for run_dir in sorted(OUTPUT_DIR.iterdir()):
        if not run_dir.is_dir():
            continue
        if task_prefix and not run_dir.name.startswith(task_prefix):
            continue
        log_path = run_dir / 'bot_log.json'
        if log_path.exists():
            with open(log_path) as f:
                data = json.load(f)
            data['_run_dir'] = str(run_dir)
            runs.append(data)
    return runs


def summary_table(df: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
    """Return a mean ± SD summary table for each metric across subjects."""
    rows = []
    for col in metrics:
        if col not in df.columns:
            rows.append({'metric': col, 'mean': float('nan'), 'sd': float('nan'), 'n': 0})
            continue
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        rows.append({
            'metric': col,
            'mean': round(series.mean(), 4),
            'sd':   round(series.std(ddof=1), 4),
            'n':    len(series),
        })
    return pd.DataFrame(rows).set_index('metric')

## 2. Load Human Data

Read the pre-processed per-subject summary CSVs from `data/human/`. Each row is one subject's aggregate statistics for a single task session.

Three exclusion flag columns are applied:
- `Session-Level Exclusions` — data quality issues for this session
- `Task-Level Exclusions`    — task-specific anomalies (e.g. floor/ceiling accuracy)
- `Subject-Level Exclusions` — cross-task exclusions (e.g. overall compliance)

Only rows where **all three** flags equal `"Include"` are retained.

In [ ]:
EXCLUSION_COLS = ['Session-Level Exclusions', 'Task-Level Exclusions', 'Subject-Level Exclusions']

def apply_exclusions(df: pd.DataFrame) -> pd.DataFrame:
    """Keep rows where all three exclusion columns equal 'Include'."""
    mask = pd.Series([True] * len(df), index=df.index)
    for col in EXCLUSION_COLS:
        if col in df.columns:
            # Normalise whitespace before comparison
            mask &= df[col].str.strip().eq('Include')
        else:
            print(f'  WARNING: exclusion column not found: {col!r}')
    return df[mask].copy()


# --- Stop Signal ---
ss_raw = pd.read_csv(STOP_SIGNAL_CSV)
ss_human = apply_exclusions(ss_raw)
print(f'Stop Signal  — raw: {len(ss_raw):,}  after exclusions: {len(ss_human):,}')

# --- Stroop ---
stroop_raw = pd.read_csv(STROOP_CSV)
stroop_human = apply_exclusions(stroop_raw)
print(f'Stroop       — raw: {len(stroop_raw):,}  after exclusions: {len(stroop_human):,}')

## 3. Load Bot Data

Scan `output/` for completed experiment bot runs. Each run directory contains a `bot_log.json` with trial-level results.

When bot runs are available they will be parsed into DataFrames with the same metric columns as the human data to enable direct comparison.

In [ ]:
# Load bot runs for each task platform
stopit_runs      = load_bot_runs('stopit_stop_signal')
expfactory_ss_runs   = load_bot_runs('expfactory_stop_signal')
cognitionrun_stroop_runs = load_bot_runs('cognitionrun_stroop')
expfactory_stroop_runs  = load_bot_runs('expfactory_stroop')

print(f'STOP-IT runs found          : {len(stopit_runs)}')
print(f'ExpFactory stop-signal runs : {len(expfactory_ss_runs)}')
print(f'Cognition.run stroop runs   : {len(cognitionrun_stroop_runs)}')
print(f'ExpFactory stroop runs      : {len(expfactory_stroop_runs)}')

if not any([stopit_runs, expfactory_ss_runs, cognitionrun_stroop_runs, expfactory_stroop_runs]):
    print('\nNo bot runs found yet. Run the experiment bot and re-execute this notebook.')

## 4. Stop Signal Metrics

Per-subject metrics computed from the human stop-signal data:

| Metric | Column | Description |
|---|---|---|
| Go RT | `go_rt` | Mean RT on go trials (ms) |
| Go Accuracy | `go_accuracy` | Proportion of go trials answered correctly |
| Go Omission Rate | `go_omission_rate` | Proportion of go trials with no response |
| Stop Failure RT | `mean_stop_failure_RT` | Mean RT on failed stop trials (ms) |
| Stop Accuracy | `stop_accuracy` | Proportion of stop trials inhibited |
| Mean SSD | `mean_SSD` | Mean stop-signal delay (ms) |

In [ ]:
SS_METRICS = [
    'go_rt',
    'go_accuracy',
    'go_omission_rate',
    'mean_stop_failure_RT',
    'stop_accuracy',
    'mean_SSD',
]

ss_summary = summary_table(ss_human, SS_METRICS)
print(f'Human stop-signal summary (n={len(ss_human)} subjects):')
ss_summary

In [ ]:
# Distribution plots for key stop-signal metrics
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Human Stop Signal — Metric Distributions', fontsize=14)

for ax, col in zip(axes.flat, SS_METRICS):
    series = pd.to_numeric(ss_human[col], errors='coerce').dropna()
    ax.hist(series, bins=40, edgecolor='white', linewidth=0.4)
    ax.set_title(col)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

## 5. Stroop Metrics

Per-subject metrics computed from the human Stroop data:

| Metric | Column | Description |
|---|---|---|
| Congruent RT | `congruent_rt` | Mean RT on congruent trials (ms) |
| Incongruent RT | `incongruent_rt` | Mean RT on incongruent trials (ms) |
| Congruent Accuracy | `congruent_accuracy` | Proportion correct on congruent trials |
| Incongruent Accuracy | `incongruent_accuracy` | Proportion correct on incongruent trials |
| Stroop Effect | _(derived)_ | incongruent_rt − congruent_rt |

In [ ]:
STROOP_METRICS = [
    'congruent_rt',
    'incongruent_rt',
    'congruent_accuracy',
    'incongruent_accuracy',
]

# Derive Stroop effect (incongruent - congruent RT) per subject
stroop_human = stroop_human.copy()
stroop_human['stroop_effect'] = (
    pd.to_numeric(stroop_human['incongruent_rt'], errors='coerce')
    - pd.to_numeric(stroop_human['congruent_rt'], errors='coerce')
)

stroop_summary = summary_table(stroop_human, STROOP_METRICS + ['stroop_effect'])
print(f'Human Stroop summary (n={len(stroop_human)} subjects):')
stroop_summary

In [ ]:
# Distribution plots for Stroop metrics
plot_cols = STROOP_METRICS + ['stroop_effect']
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Human Stroop — Metric Distributions', fontsize=14)

for ax, col in zip(axes.flat, plot_cols):
    series = pd.to_numeric(stroop_human[col], errors='coerce').dropna()
    ax.hist(series, bins=40, edgecolor='white', linewidth=0.4)
    ax.set_title(col)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')

# Hide unused subplot
axes.flat[-1].set_visible(False)

plt.tight_layout()
plt.show()

## 6. Cross-Platform Comparison

**Placeholder** — this section will compare human vs. bot metrics once STOP-IT and cognition.run bot runs are available.

Planned comparisons:
- Human vs. STOP-IT bot: go RT, stop accuracy, mean SSD
- Human vs. ExpFactory stop-signal bot: same metrics
- Human vs. cognition.run Stroop bot: congruent/incongruent RT, Stroop effect
- Human vs. ExpFactory Stroop bot: same metrics

Each comparison will use a table of mean ± SD and a side-by-side distribution plot.

In [ ]:
# TODO: populate once bot runs exist
# Expected structure of bot_log.json:
#   {
#     "task": "stop_signal" | "stroop",
#     "platform": "stopit" | "expfactory" | "cognitionrun",
#     "trials": [
#       {"trial_type": "go"|"stop", "rt": float, "correct": bool, "ssd": float|null, ...},
#       ...
#     ],
#     "summary": {<same columns as human CSV>}
#   }
#
# When bot runs are available, load with load_bot_runs() and extract
# the 'summary' dict from each run to build a bot_df comparable to
# ss_human / stroop_human, then call summary_table() for comparison.

if stopit_runs:
    # Example: bot_df = pd.DataFrame([r['summary'] for r in stopit_runs])
    # bot_summary = summary_table(bot_df, SS_METRICS)
    pass
else:
    print('No bot data available yet — re-run after completing bot sessions.')